# Week 07 실습 · 클래스와 객체지향 설계

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MNU-AI-Programming/AI-Assisted-Python-Programming/blob/main/weeks/week07/practice.ipynb)

**AI활용코딩연습(ZBE135)** · 이영호 교수 · 국립목포대학교 컴퓨터학부

## 오늘의 목표

1. 딕셔너리 + 함수 방식의 한계를 보고, **상태(데이터)와 동작(메서드)을 묶는** 클래스가 필요한 이유를 이해합니다.
2. `class`, `__init__`, `self`, 메서드, `__str__`/`__repr__`, `@property`를 사용합니다.
3. 메서드로만 상태를 바꾸게 하여 **잘못된 상태**(대출 중인 책을 또 빌려줌 등)를 막습니다.
4. 여러 클래스가 협력하는 **도서 대출 관리 시스템**을 만들고 테스트합니다.
5. 상속과 `@dataclass`를 맛보고, AI 코드의 **클래스 변수 공유 버그**를 찾아냅니다.

| 구간 | 내용 | 시간 |
|---|---|---|
| 1 | 왜 클래스인가? | 10분 |
| 2 | Level 1 클래스의 기본 | 25분 |
| 3 | Level 2 바꾸기 | 10분 |
| 4 | Level 3 도서 대출 관리 시스템 | 35분 |
| 5 | 상속과 dataclass | 15분 |
| 6 | Level 4 AI 코드의 클래스 변수 | 15분 |
| 7 | 정리 | 10분 |

---
## 1. 왜 클래스인가?

지금까지 배운 방식(딕셔너리 + 함수)으로 책 대출을 관리해 봅시다.

In [ ]:
book = {"id": "B001", "title": "파이썬 첫걸음", "on_loan": False}

def borrow(book):
    book["on_loan"] = True

borrow(book)
borrow(book)                 # 이미 대출 중인데 또 빌려줌 → 아무도 막지 않는다
book["on_loan"] = "네"        # 누구나 아무 값이나 넣을 수 있다
book["tittle"] = "오타"       # 오타 키가 새로 생겨도 모른다
print(book)

**문제점**

- 데이터(딕셔너리)와 그 데이터를 다루는 규칙(함수)이 **따로** 있습니다.
- 규칙을 거치지 않고 데이터를 **직접 바꿀 수** 있습니다.

**클래스**는 데이터와 규칙을 하나로 묶은 **설계도**이고, 그 설계도로 만든 것이 **객체(인스턴스)** 입니다.

| 용어 | 비유 | 예 |
|---|---|---|
| 클래스 | 붕어빵 틀 | `Book` |
| 객체(인스턴스) | 붕어빵 | `Book("B001", "파이썬 첫걸음")` |
| 속성(attribute) | 객체가 가진 데이터 | `book.title`, `book.on_loan` |
| 메서드(method) | 객체가 하는 동작 | `book.borrow()` |

---
## 2. Level 1 따라하기: 클래스의 기본

In [ ]:
class Book:
    """도서 한 권."""

    def __init__(self, book_id, title, author="미상"):
        # __init__: 객체를 만들 때 자동으로 실행되어 속성을 준비한다
        self.book_id = book_id
        self.title = title
        self.author = author
        self.on_loan = False

    def borrow(self):
        if self.on_loan:
            raise ValueError(f"이미 대출 중: {self.title}")
        self.on_loan = True

    def give_back(self):
        if not self.on_loan:
            raise ValueError(f"대출 중이 아님: {self.title}")
        self.on_loan = False


b1 = Book("B001", "파이썬 첫걸음", "김목포")
b2 = Book("B002", "데이터 분석 입문")
print(b1.title, b1.author, b1.on_loan)
print(b2.title, b2.author, b2.on_loan)

### `self`란?

`b1.borrow()`를 호출하면 Python은 실제로 `Book.borrow(b1)`을 실행합니다. 즉 **`self`는 메서드를 호출한 그 객체**입니다. 그래서 `b1`과 `b2`는 각자 자기 `on_loan`을 가집니다.

In [ ]:
b1.borrow()
print("b1:", b1.on_loan, "/ b2:", b2.on_loan)

try:
    b1.borrow()          # 규칙이 클래스 안에 있으니 잘못된 대출을 막는다
except ValueError as e:
    print("ValueError:", e)

b1.give_back()
print("반납 후 b1:", b1.on_loan)

### ❓ 예상해 보기: 객체를 그냥 출력하면?

In [ ]:
print(b1)

`<__main__.Book object at 0x...>`처럼 알아보기 어렵습니다. 특별한 메서드를 정의하면 출력 모양을 정할 수 있습니다.

| 메서드 | 언제 쓰이나 | 목적 |
|---|---|---|
| `__str__` | `print(obj)`, `str(obj)` | 사람이 읽기 좋은 설명 |
| `__repr__` | 노트북에서 값 표시, 리스트 안 출력 | 개발자용 정확한 표현 |

In [ ]:
class Book:
    """도서 한 권."""

    def __init__(self, book_id, title, author="미상"):
        self.book_id = book_id
        self.title = title
        self.author = author
        self.on_loan = False

    def borrow(self):
        if self.on_loan:
            raise ValueError(f"이미 대출 중: {self.title}")
        self.on_loan = True

    def give_back(self):
        if not self.on_loan:
            raise ValueError(f"대출 중이 아님: {self.title}")
        self.on_loan = False

    @property
    def status(self):
        """계산되는 속성: 괄호 없이 book.status로 읽는다."""
        return "대출 중" if self.on_loan else "대출 가능"

    def __str__(self):
        return f"[{self.book_id}] {self.title} ({self.author}) - {self.status}"

    def __repr__(self):
        return f"Book({self.book_id!r}, {self.title!r})"


b1 = Book("B001", "파이썬 첫걸음", "김목포")
b1.borrow()
print(b1)
print([b1, Book("B002", "데이터 분석 입문")])
print(b1.status)

---
## 3. Level 2 바꾸기

### ✏️ 직접 해보기

1. `Book`에 `loan_count`(누적 대출 횟수) 속성을 추가하고, `borrow()`할 때마다 1씩 늘려 보세요.
2. `__str__`에 누적 대출 횟수를 함께 보여 주세요.
3. 아래 셀에서 세 번 빌리고 반납한 뒤 출력해 확인합니다.

In [ ]:
class CountingBook(Book):        # Book을 물려받아 필요한 것만 바꾼다 (5절에서 자세히)
    def __init__(self, book_id, title, author="미상"):
        super().__init__(book_id, title, author)
        self.loan_count = 0

    def borrow(self):
        super().borrow()
        self.loan_count += 1     # ← 직접 바꿔 보세요: 예) 대출 횟수 제한 추가

    def __str__(self):
        return super().__str__() + f" / 누적 {self.loan_count}회"


cb = CountingBook("B010", "알고리즘 산책")
for _ in range(3):
    cb.borrow()
    cb.give_back()
print(cb)

---
## 4. Level 3 적용하기: 도서 대출 관리 시스템

**요구사항 (가상 규정)**

1. 회원은 최대 **3권**까지 빌릴 수 있습니다.
2. 대출 기간은 **14일**, 연체료는 **하루 100원**입니다.
3. 존재하지 않는 회원·책은 `KeyError`, 규칙 위반은 `ValueError`로 알립니다.
4. 오늘 기준 연체 중인 대출 목록을 볼 수 있습니다.

**설계**: 누가 무엇을 책임지는가?

| 클래스 | 상태(속성) | 책임(메서드) |
|---|---|---|
| `Book` | id, 제목, 대출 여부 | 대출/반납 상태 바꾸기 |
| `Member` | 이름, 빌린 책 id 목록 | 한도 확인 |
| `Library` | 책 목록, 회원 목록, 대출 기록(반납 예정일) | 대출·반납·연체 조회 (규칙 조율) |

In [ ]:
from datetime import date, timedelta


class Member:
    MAX_BOOKS = 3                      # 클래스 변수: 모든 회원이 공유하는 '상수'

    def __init__(self, name):
        self.name = name
        self.borrowed = []             # 인스턴스 변수: 회원마다 따로

    def can_borrow(self):
        return len(self.borrowed) < self.MAX_BOOKS

    def __repr__(self):
        return f"Member({self.name!r}, borrowed={self.borrowed})"


class Library:
    LOAN_DAYS = 14
    FINE_PER_DAY = 100

    def __init__(self):
        self.books = {}                # book_id → Book
        self.members = {}              # name → Member
        self.loans = {}                # book_id → (member name, due date)

    def add_book(self, book):
        self.books[book.book_id] = book

    def join(self, member):
        self.members[member.name] = member

    def borrow(self, name, book_id, today):
        member = self.members[name]            # 없으면 KeyError
        book = self.books[book_id]             # 없으면 KeyError
        if not member.can_borrow():
            raise ValueError(f"{name}: 대출 한도 {Member.MAX_BOOKS}권 초과")
        book.borrow()                          # 이미 대출 중이면 Book이 ValueError
        member.borrowed.append(book_id)
        due = today + timedelta(days=self.LOAN_DAYS)
        self.loans[book_id] = (name, due)
        return due

    def give_back(self, book_id, today):
        if book_id not in self.loans:
            raise ValueError(f"대출 기록 없음: {book_id}")
        name, due = self.loans.pop(book_id)
        self.books[book_id].give_back()
        self.members[name].borrowed.remove(book_id)
        late_days = max(0, (today - due).days)
        return late_days * self.FINE_PER_DAY

    def overdue(self, today):
        """(책 제목, 회원, 연체 일수) 목록, 연체가 긴 순."""
        result = [(self.books[bid].title, name, (today - due).days)
                  for bid, (name, due) in self.loans.items() if today > due]
        return sorted(result, key=lambda r: -r[2])

In [ ]:
lib = Library()
for bid, title in [("B001", "파이썬 첫걸음"), ("B002", "데이터 분석 입문"), ("B003", "알고리즘 산책"),
                   ("B004", "통계 이야기"), ("B005", "AI와 함께 코딩")]:
    lib.add_book(Book(bid, title))
for name in ["김민준", "이서연"]:
    lib.join(Member(name))

d0 = date(2026, 9, 1)
print("반납 예정:", lib.borrow("김민준", "B001", d0))
lib.borrow("김민준", "B002", d0)
lib.borrow("이서연", "B003", d0 + timedelta(days=5))
print(lib.members["김민준"], lib.members["이서연"])

In [ ]:
# 규칙 위반은 정확한 오류로 알린다
for args in [("김민준", "B001", d0), ("박지호", "B004", d0), ("이서연", "B999", d0)]:
    try:
        lib.borrow(*args)
    except (ValueError, KeyError) as e:
        print(f"{type(e).__name__}: {e}")

lib.borrow("김민준", "B004", d0)
try:
    lib.borrow("김민준", "B005", d0)      # 4권째
except ValueError as e:
    print("ValueError:", e)

In [ ]:
today = date(2026, 9, 20)
print("연체 목록:", lib.overdue(today))
print("B001 반납 연체료:", lib.give_back("B001", today), "원")
print("B003 반납 연체료:", lib.give_back("B003", today), "원")
print("남은 대출:", {k: (v[0], v[1].isoformat()) for k, v in lib.loans.items()})

### 테스트: 시나리오로 확인하기

클래스는 **상태가 이어지기 때문에** 여러 동작을 순서대로 실행하는 시나리오 테스트가 중요합니다.
테스트마다 **새 `Library`를 만들어** 다른 테스트의 영향을 받지 않게 합니다.

In [ ]:
def make_library():
    lib = Library()
    for i in range(1, 6):
        lib.add_book(Book(f"B00{i}", f"책{i}"))
    lib.join(Member("가"))
    lib.join(Member("나"))
    return lib


def test_limit_and_independent_members():
    lib = make_library()
    d = date(2026, 9, 1)
    for bid in ["B001", "B002", "B003"]:
        lib.borrow("가", bid, d)
    try:
        lib.borrow("가", "B004", d)
        assert False, "4권째는 ValueError"
    except ValueError:
        pass
    lib.borrow("나", "B004", d)                 # 다른 회원은 영향 없음
    assert lib.members["나"].borrowed == ["B004"]


def test_fine_boundary():
    lib = make_library()
    d = date(2026, 9, 1)
    lib.borrow("가", "B001", d)
    assert lib.give_back("B001", d + timedelta(days=14)) == 0      # 반납 예정일 당일
    lib.borrow("가", "B001", d)
    assert lib.give_back("B001", d + timedelta(days=15)) == 100    # 하루 연체


def test_cannot_return_twice():
    lib = make_library()
    lib.borrow("가", "B001", date(2026, 9, 1))
    lib.give_back("B001", date(2026, 9, 2))
    try:
        lib.give_back("B001", date(2026, 9, 2))
        assert False
    except ValueError:
        pass


for t in [test_limit_and_independent_members, test_fine_boundary, test_cannot_return_twice]:
    t()
    print("✅", t.__name__)

---
## 5. 상속과 dataclass

### 상속: 공통은 물려받고, 다른 부분만 바꾼다

전자책은 여러 사람이 동시에 빌릴 수 있다고 해 봅시다. `Book`을 **상속**해서 `borrow`/`give_back`만 바꿉니다(**오버라이딩**).

In [ ]:
class EBook(Book):
    def __init__(self, book_id, title, author="미상", licenses=3):
        super().__init__(book_id, title, author)     # 부모의 __init__ 재사용
        self.licenses = licenses
        self.active = 0

    def borrow(self):
        if self.active >= self.licenses:
            raise ValueError(f"동시 대출 한도 {self.licenses} 초과: {self.title}")
        self.active += 1
        self.on_loan = self.active == self.licenses

    def give_back(self):
        if self.active == 0:
            raise ValueError(f"대출 중이 아님: {self.title}")
        self.active -= 1
        self.on_loan = False


eb = EBook("E001", "전자책으로 배우는 파이썬", licenses=2)
eb.borrow(); eb.borrow()
print(eb, "/ isinstance(eb, Book):", isinstance(eb, Book))

**다형성**: `Library`는 `Book`이든 `EBook`이든 똑같이 `book.borrow()`를 부르기만 하면 됩니다. 각 객체가 자기 방식대로 동작합니다.

### `@dataclass`: 데이터 중심 클래스를 짧게

속성만 주로 담는 클래스는 `@dataclass`가 `__init__`, `__repr__`, `__eq__`를 자동으로 만들어 줍니다.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class LoanRecord:
    book_id: str
    member: str
    due: date
    returned: bool = False
    notes: list = field(default_factory=list)    # 가변 기본값은 이렇게!


r1 = LoanRecord("B001", "김민준", date(2026, 9, 15))
r2 = LoanRecord("B001", "김민준", date(2026, 9, 15))
print(r1)
print("값이 같으면 같다:", r1 == r2)

---
## 6. Level 4 AI와 함께: 클래스 변수 공유 버그

"회원 클래스를 만들어 줘. 회원마다 빌린 책 목록이 있어야 해"라고 요청해 받은 코드입니다.

In [ ]:
class MemberAI:
    borrowed = []                     # 클래스 몸체에 바로 쓴 리스트

    def __init__(self, name):
        self.name = name

    def add(self, book_id):
        self.borrowed.append(book_id)


m1, m2 = MemberAI("김민준"), MemberAI("이서연")
m1.add("B001")
print("이서연의 목록:", m2.borrowed)
print("같은 리스트?", m1.borrowed is m2.borrowed)

### 🔎 원인과 해결

- 클래스 몸체에 쓴 `borrowed = []`는 **클래스 변수**로, 모든 인스턴스가 **하나의 리스트를 공유**합니다.
- 회원마다 따로 가져야 하는 값은 `__init__` 안에서 `self.borrowed = []`로 만듭니다. (5주차의 가변 기본값 문제와 같은 뿌리)
- 클래스 변수는 `MAX_BOOKS = 3`처럼 **모두가 공유하는 상수**에 씁니다.

In [ ]:
def test_members_have_separate_lists(cls):
    a, b = cls("가"), cls("나")
    a.borrowed.append("B001") if not hasattr(a, "add") else a.add("B001")
    return b.borrowed == [] and a.borrowed is not b.borrowed

print("MemberAI 통과?", test_members_have_separate_lists(MemberAI))
print("Member  통과?", test_members_have_separate_lists(Member))

### 📝 AI에게 클래스를 요청할 때

```text
회원(Member) 클래스를 만들어 주세요.
- 회원마다 따로 가져야 하는 리스트·딕셔너리는 __init__ 안에서 self로 만들 것
- 상태 변경은 메서드로만 하고, 규칙 위반은 ValueError로 알릴 것
- 두 회원 객체가 서로 영향을 주지 않는지 확인하는 테스트를 포함할 것
```

---
## 7. 정리

| 오늘 배운 것 | 핵심 |
|---|---|
| 클래스와 객체 | 설계도와 실체, 상태(속성) + 동작(메서드) |
| `__init__`, `self` | 객체를 준비하는 메서드, 호출한 그 객체 |
| 캡슐화 | 상태는 메서드로만 바꾸고, 규칙 위반은 `ValueError` |
| 특별 메서드 | `__str__`(사람용), `__repr__`(개발자용), `@property` |
| 협력 | 책임 나누기: `Book` · `Member` · `Library` |
| 상속 | `class EBook(Book)`, `super()`, 오버라이딩, 다형성 |
| dataclass | 데이터 중심 클래스, 가변 기본값은 `field(default_factory=list)` |
| AI 검증 | 클래스 변수 리스트 공유 → `__init__`에서 `self.x = []` |

### 자기 점검

- [ ] `b1.borrow()`에서 `self`가 무엇인지 설명할 수 있다.
- [ ] 딕셔너리 대신 클래스를 쓰면 막을 수 있는 실수를 두 가지 말할 수 있다.
- [ ] 클래스 변수와 인스턴스 변수의 차이를 예로 설명할 수 있다.
- [ ] 시나리오 테스트를 만들 때 테스트마다 새 객체를 만드는 이유를 안다.

### 다음 주 예고

**Week 08 · 중간 실기**: 1~7주차 내용을 통합해 성적 분석 미니 프로젝트를 해결합니다. 이번 주 실습 노트북과 과제를 꼭 복습하세요.